In [1]:
# groq api key
# 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
import os
os.environ["GROQ_API_KEY"] = "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"

import os
import pandas as pd
import numpy as np

import nest_asyncio
nest_asyncio.apply()


In [27]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.groq import Groq

# Load documents from the /data directory
documents = SimpleDirectoryReader('./data').load_data()

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Initialize the embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
# embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")

# Build a simple vector index
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

# Set up the LLM
llm = Groq(model="llama3-8b-8192")



In [28]:
from llama_index.core.query_engine import RetrieverQueryEngine
from crewai.tools import BaseTool
from crewai.tools import tool

# Create a query engine
query_engine = index.as_query_engine(llm=llm)


# Define a function tool agents can use
@tool("Medical Evidence")
def query_medical_evidence(question: str) -> str:
    """This tool is used to query medical documents and return evidence for a query from the documents."""
    response = query_engine.query(question)
    return str(response)


In [29]:
from crewai import Agent, Task, Crew

# Research Agent - Finds evidence
research_agent = Agent(
    role="Medical Researcher",
    goal="Find relevant medical evidence from patient documents",
    tools=[query_medical_evidence],
    llm=llm,
    backstory="An experienced medical researcher specializing in extracting critical patient information.",
)

# Summarizer Agent - Summarizes findings
summarizer_agent = Agent(
    role="Medical Summarizer",
    goal="Summarize key points from the research findings clearly and concisely",
    tools=[query_medical_evidence],
    llm=llm,
    backstory="A detail-oriented medical writer specializing in patient summary reports.",
)

# Verifier Agent - Verifies correctness
verifier_agent = Agent(
    role="Evidence Verifier",
    goal="Cross-check facts in the medical summary with source documents",
    tools=[query_medical_evidence],
    llm=llm,
    backstory="A meticulous medical reviewer ensuring factual accuracy.",
)

# Final Report Agent - Compiles the report
report_agent = Agent(
    role="Report Compiler",
    goal="Combine verified summaries into a final formatted medical report",
    tools=[],
    llm=llm,
    backstory="An experienced medical editor organizing verified content into polished reports.",
)



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



In [30]:
# Tasks each agent must perform
research_task = Task(
    description="Find and extract evidence about the patient's disease history, medications, and \
    treatments from the available documents.",
    agent=research_agent,
    expected_output='A refined finalized version of extracted evidence that must contain patient disease history, medications, and \
    treatments.',
)

summarize_task = Task(
    description="Summarize the extracted medical evidence clearly, covering main points.",
    agent=summarizer_agent,
    expected_output='A summarized version of extracted evidence.',
)

verify_task = Task(
    description="Verify the summarized findings by checking them against original documents.",
    agent=verifier_agent,
    expected_output='A verified and corrected version of extracted evidence.',
)

compile_task = Task(
    description="Compile the verified summaries into a clean and structured final report.",
    agent=report_agent,
    expected_output='A final refined version of extracted evidence.',
)


In [31]:
import litellm
litellm._turn_on_debug()

from crewai import LLM

def getLLM():
    return LLM(
        model=llm,
        api_version="version",
        api_key="key",
        base_url="base-url"
    )
# litellm.set_verbose = True  # Enable verbose mode

# Create the crew
crew = Crew(
    agents=[research_agent, summarizer_agent, verifier_agent, report_agent],
    tasks=[research_task, summarize_task, verify_task, compile_task],
    verbose=True,
    # llm=llm,
)

# Run the Crew
final_output = crew.kickoff()

print("\n\n📝 FINAL REPORT:")
print(final_output)



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 571ea66a-e0db-4c50-811c-492dc174276c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



🚀 Crew: crew
└── 📋 Task: a4a3147e-8031-4774-b82b-2f691daeb07c
       Status: Executing Task...


Provider List: https://docs.litellm.ai/docs/providers



...

RAW RESPONSE:
{"id": "chatcmpl-0ab7df69-4c9c-4d58-b43b-92dff08529b0", "object": "chat.completion", "created": 1745990629, "model": "llama3-8b-8192", "choices": [{"index": 0, "message": {"role": "assistant", "content": ""}, "logprobs": null, "finish_reason": "stop"}], "usage": {"queue_time": -9223372036.979815, "prompt_tokens": 996, "prompt_time": 0.125038129, "completion_tokens": 1, "completion_time": 0.000833333, "total_tokens": 997, "total_time": 0.125871462}, "usage_breakdown": {"models": null}, "system_fingerprint": "fp_dadc9d6142", "x_groq": {"id": "req_01jt2jde84e8asmjz20zvs8fqh"}}


 Received None or empty response from LLM call.
 An unknown error occurred. Please check the details below.
 Error details: Invalid response from LLM call - None or empty.
 An unknown error occurred. Please check the details below.
 Error details: Invalid response from LLM call - None or empty.


🚀 Crew: crew
└── 📋 Task: a4a3147e-8031-4774-b82b-2f691daeb07c
       Assigned to: Medical Researcher
       Status: ❌ Failed
    ├── 🤖 Agent: Medical Researcher
    │       Status: In Progress
    ├── 🤖 Agent: Medical Researcher
    │       Status: In Progress
    └── 🤖 Agent: Medical Researcher
            Status: In Progress

╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: a4a3147e-8031-4774-b82b-2f691daeb07c                                                                     │
│  Agent: Medical Researcher                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 571ea66a-e0db-4c50-811c-492dc174276c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ValueError: Invalid response from LLM call - None or empty.